# Kaggle training — Akkadian → English (ByT5)

Вариант `colab_train.ipynb` для Kaggle Notebooks (квота 30 GPU-ч/нед, сессии до 12 ч,
фоновое выполнение через Save & Run All). Чекпоинты пушатся на HF Hub, резюм после обрыва — автоматический.

**Настройка (один раз):**
1. Справа **Session options → Accelerator → GPU P100** (или T4 x2) и **Internet → On**
   (для интернета нужен подтверждённый телефон в профиле Kaggle).
2. Справа **Input → + Add Input** → вкладка Competitions → **Deep Past Initiative: Machine Translation**.
3. Меню **Add-ons → Secrets** → добавь `HF_TOKEN` (тип Write) и `WANDB_API_KEY`, включи галочки Attach.

(`KAGGLE_USERNAME`/`KAGGLE_KEY` тут не нужны — данные уже подключены как Input.)

In [ ]:
CONFIG = "configs/baseline.yaml"  # <- какой эксперимент запускаем
BRANCH = "ml-dev"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # на T4 x2 учимся на одной GPU, без DataParallel
!nvidia-smi -L

In [ ]:
!rm -rf /kaggle/working/repo
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /kaggle/working/repo
%cd /kaggle/working/repo/ml
!pip install -q -e . sacrebleu

In [ ]:
# секреты -> окружение; имя HF-репозитория выводим из токена и конфига
import yaml
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        print("no secret:", key)

from huggingface_hub import whoami
HF_USER = whoami(os.environ["HF_TOKEN"])["name"]
RUN_NAME = yaml.safe_load(open(CONFIG))["run_name"]
HUB_ID = f"{HF_USER}/akkadian-{RUN_NAME}"
FINAL = yaml.safe_load(open(CONFIG))["output_dir"] + "/final"
print("чекпоинты ->", HUB_ID)

In [ ]:
# данные соревнования уже подключены как Input — копируем нужные csv
COMP = "/kaggle/input/deep-past-initiative-machine-translation"
!mkdir -p data && cp {COMP}/train.csv {COMP}/test.csv \
    {COMP}/published_texts.csv {COMP}/Sentences_Oare_FirstWord_LinNum.csv data/
!python -m akkadian_nmt.data_prep --data_dir=./data --out_dir=./data/processed

In [ ]:
# обучение: чекпоинты пушатся на Hub, резюм с Hub после обрыва — автоматический
!python -m akkadian_nmt.train --config={CONFIG} \
    --push_to_hub=True --hub_model_id={HUB_ID}

In [ ]:
# оценка на dev: greedy vs beam {1,4,8}
!python -m akkadian_nmt.evaluate beam_sweep --model_dirs={FINAL} --max_samples=200

In [ ]:
# полный метрический набор (BLEU + chrF++ + geo-mean + COMET) на лучшей конфигурации
!pip install -q unbabel-comet
!python -m akkadian_nmt.evaluate run --model_dirs={FINAL} --num_beams=4 --comet=True \
    --out_file=data/dev_predictions.json

In [ ]:
# предсказания для сабмита -> data/results.csv (скачай из Output справа
# или сабмить прямо с Kaggle: вкладка Output -> Submit to competition)
!python model.py predict-file --dataset=data/test.csv --model_dir={FINAL}
!head -3 data/results.csv